In [18]:
import os

from enum import Enum
from typing import Protocol

from dotenv import load_dotenv
from langchain_groq import ChatGroq

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    model_validator,
)

print("Imports loaded successfully.")

Imports loaded successfully.


In [19]:
load_dotenv(
    dotenv_path=r"C:\Users\Acer\lab\.env",
    override=True,
)

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY was not found."
    )

GROQ_MODEL = os.getenv(
    "GROQ_MODEL",
    "openai/gpt-oss-120b",
)

print("Groq API key loaded:", bool(api_key))
print("Groq model:", GROQ_MODEL)

Groq API key loaded: True
Groq model: openai/gpt-oss-120b


In [20]:
class Intent(str, Enum):
    NAVIGATE = "NAVIGATE"
    CLARIFY = "CLARIFY"
    UNKNOWN = "UNKNOWN"


class DestinationType(str, Enum):
    DEPARTMENT = "DEPARTMENT"
    WARD = "WARD"


print("Enums defined successfully.")

Enums defined successfully.


In [21]:
class DestinationDecision(BaseModel):
    """
    Structured output from Teammate A -> Teammate B.

    Department example:
        destination_type = DEPARTMENT
        destination = "radiology"

    Ward example:
        destination_type = WARD
        destination = "ward_0123"
        ward_number = 123
    """

    model_config = ConfigDict(
        extra="forbid",
        frozen=True,
    )

    intent: Intent

    destination_type: DestinationType | None = None

    destination: str | None = None

    ward_number: int | None = Field(
        default=None,
        ge=1,
        le=5000,
    )

    confidence: float = Field(
        ge=0.0,
        le=1.0,
    )

    needs_clarification: bool

    candidates: list[str] = Field(
        default_factory=list
    )

    visitor_message: str = Field(
        min_length=1,
        max_length=180,
    )


    @model_validator(mode="after")
    def validate_semantics(self):

        supported_departments = {
            "radiology",
            "pharmacy",
            "consultation_room",
            "elevator",
            "restroom",
            "reception",
        }

        # ====================================================
        # NAVIGATE
        # ====================================================

        if self.intent is Intent.NAVIGATE:

            if self.destination is None:
                raise ValueError(
                    "NAVIGATE requires a destination."
                )

            if self.needs_clarification:
                raise ValueError(
                    "NAVIGATE cannot require clarification."
                )

            if self.candidates:
                raise ValueError(
                    "NAVIGATE cannot contain candidates."
                )

            # ------------------------------
            # WARD
            # ------------------------------

            if self.destination_type is DestinationType.WARD:

                if self.ward_number is None:
                    raise ValueError(
                        "WARD requires ward_number."
                    )

                expected_destination = (
                    f"ward_{self.ward_number:04d}"
                )

                if self.destination != expected_destination:
                    raise ValueError(
                        "Ward destination must be "
                        f"{expected_destination}."
                    )

            # ------------------------------
            # DEPARTMENT
            # ------------------------------

            elif (
                self.destination_type
                is DestinationType.DEPARTMENT
            ):

                if self.destination not in supported_departments:
                    raise ValueError(
                        "Unsupported department destination."
                    )

                if self.ward_number is not None:
                    raise ValueError(
                        "Department destination cannot "
                        "contain ward_number."
                    )

            else:
                raise ValueError(
                    "NAVIGATE requires destination_type "
                    "DEPARTMENT or WARD."
                )

        # ====================================================
        # CLARIFY
        # ====================================================

        elif self.intent is Intent.CLARIFY:

            if self.destination is not None:
                raise ValueError(
                    "CLARIFY cannot select a destination."
                )

            if self.destination_type is not None:
                raise ValueError(
                    "CLARIFY cannot select a destination type."
                )

            if self.ward_number is not None:
                raise ValueError(
                    "CLARIFY cannot select a ward number."
                )

            if not self.needs_clarification:
                raise ValueError(
                    "CLARIFY requires "
                    "needs_clarification=True."
                )

        # ====================================================
        # UNKNOWN
        # ====================================================

        elif self.intent is Intent.UNKNOWN:

            if self.destination is not None:
                raise ValueError(
                    "UNKNOWN cannot select a destination."
                )

            if self.destination_type is not None:
                raise ValueError(
                    "UNKNOWN cannot select a destination type."
                )

            if self.ward_number is not None:
                raise ValueError(
                    "UNKNOWN cannot select a ward number."
                )

            if self.needs_clarification:
                raise ValueError(
                    "UNKNOWN cannot require clarification."
                )

            if self.candidates:
                raise ValueError(
                    "UNKNOWN cannot contain candidates."
                )

        return self


print("DestinationDecision schema defined successfully.")

DestinationDecision schema defined successfully.


In [22]:
DEPARTMENTS = {
    "radiology": (
        "Radiology department for X-rays, scans, "
        "and medical imaging."
    ),

    "pharmacy": (
        "Pharmacy where visitors collect "
        "prescribed medication."
    ),

    "consultation_room": (
        "Consultation room where patients attend "
        "medical consultations."
    ),

    "elevator": (
        "Public elevator or lift."
    ),

    "restroom": (
        "Public restroom or toilet facilities."
    ),

    "reception": (
        "Hospital reception or front desk."
    ),
}


WARD_MIN = 1
WARD_MAX = 5000


print("Departments:", list(DEPARTMENTS.keys()))
print(
    f"Supported wards: "
    f"{WARD_MIN:04d} - {WARD_MAX:04d}"
)

Departments: ['radiology', 'pharmacy', 'consultation_room', 'elevator', 'restroom', 'reception']
Supported wards: 0001 - 5000


In [23]:
SYSTEM_PROMPT = f"""
You are the destination-understanding component
of a hospital navigation robot.

Your ONLY responsibility is to understand where
the visitor wants to go.

You must return a DestinationDecision.


SUPPORTED DEPARTMENTS

{DEPARTMENTS}


SUPPORTED WARDS

The hospital contains wards numbered:

Ward 0001 through Ward 5000.

Wards are parameterized destinations.

Do NOT enumerate all 5000 wards.


WARD NORMALIZATION

Convert ward numbers into four-digit canonical IDs.

Examples:

Ward 1
-> ward_number = 1
-> destination = "ward_0001"

Ward 27
-> ward_number = 27
-> destination = "ward_0027"

Ward 123
-> ward_number = 123
-> destination = "ward_0123"

Ward 1234
-> ward_number = 1234
-> destination = "ward_1234"

Ward 5000
-> ward_number = 5000
-> destination = "ward_5000"


INTENT RULES


NAVIGATE

Return NAVIGATE when exactly one valid destination
can be determined.

For a department:

destination_type = "DEPARTMENT"

For a ward:

destination_type = "WARD"


CLARIFY

Return CLARIFY when the visitor has not provided
enough information to identify the destination.

Example:

"Take me to the ward."

The ward number is missing.

Ask:

"Which ward number would you like to visit?"


UNKNOWN

Return UNKNOWN when the requested destination
is outside the supported hospital map.

Examples:

"Take me to Starbucks."

-> UNKNOWN

"Take me to Ward 5001."

-> UNKNOWN


IMPORTANT RULES

- Use semantic understanding.
- Do not rely only on exact keyword matching.
- Never invent destinations.
- Never guess missing ward numbers.
- Never clamp invalid ward numbers.
- Ward numbers must be between 1 and 5000.
- Only use supported department IDs.
- Keep visitor_message short and natural.
- confidence must be between 0 and 1.
- confidence is metadata only.
- Do not expose hidden reasoning.
""".strip()


print("System prompt created.")

System prompt created.


In [24]:
class DestinationModel(Protocol):

    def decide(
        self,
        user_text: str,
    ) -> DestinationDecision:
        ...


print("DestinationModel protocol defined.")

DestinationModel protocol defined.


In [25]:
class GroqDestinationModel:

    def __init__(
        self,
        api_key: str,
        model_name: str,
    ):

        if not api_key.strip():
            raise ValueError(
                "GROQ_API_KEY is empty."
            )

        chat = ChatGroq(
            api_key=api_key,
            model=model_name,
            temperature=0,
            timeout=30,
            max_retries=1,
        )

        self.structured_model = (
            chat.with_structured_output(
                DestinationDecision
            )
        )


    def decide(
        self,
        user_text: str,
    ) -> DestinationDecision:

        result = self.structured_model.invoke([
            (
                "system",
                SYSTEM_PROMPT,
            ),
            (
                "human",
                user_text,
            ),
        ])

        return result


print("GroqDestinationModel defined.")

GroqDestinationModel defined.


In [26]:
class DestinationResolver:

    def __init__(
        self,
        destination_model: DestinationModel,
    ):

        self.destination_model = (
            destination_model
        )


    def resolve_destination(
        self,
        user_text: str,
    ) -> DestinationDecision:

        cleaned = " ".join(
            user_text.split()
        )

        # Empty request
        if not cleaned:

            return DestinationDecision(
                intent=Intent.CLARIFY,
                destination_type=None,
                destination=None,
                ward_number=None,
                confidence=1.0,
                needs_clarification=True,
                candidates=[],
                visitor_message=(
                    "Where would you like me "
                    "to take you?"
                ),
            )

        # Bound input size
        if len(cleaned) > 500:

            raise ValueError(
                "Visitor instruction is too long."
            )

        return (
            self.destination_model.decide(
                cleaned
            )
        )


print("DestinationResolver defined.")

DestinationResolver defined.


In [27]:
groq_destination_model = (
    GroqDestinationModel(
        api_key=api_key,
        model_name=GROQ_MODEL,
    )
)


resolver = DestinationResolver(
    destination_model=groq_destination_model
)


print(
    "Agent A initialized successfully."
)

print(
    "Resolver:",
    type(resolver).__name__,
)

print(
    "Model:",
    type(groq_destination_model).__name__,
)

Agent A initialized successfully.
Resolver: DestinationResolver
Model: GroqDestinationModel


In [28]:
decision = resolver.resolve_destination(
    "I want to visit my uncle at Ward 123."
)


print(
    decision.model_dump_json(
        indent=2
    )
)

{
  "intent": "NAVIGATE",
  "destination_type": "WARD",
  "destination": "ward_0123",
  "ward_number": 123,
  "confidence": 1.0,
  "needs_clarification": false,
  "candidates": [],
  "visitor_message": "Sure, taking you to Ward 123."
}


In [29]:
"""WARD_TESTS = [
    "Take me to Ward 1.",
    "Please bring me to Ward 0001.",
    "I need Ward 27.",
    "Take me to Ward 1234.",
    "Please guide me to Ward 5000.",
    "Take me to the ward.",
    "Take me to Ward 5001.",
]


for text in WARD_TESTS:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "INPUT:",
        text,
    )

    try:

        result = (
            resolver.resolve_destination(
                text
            )
        )

        print(
            result.model_dump_json(
                indent=2
            )
        )

    except Exception as exc:

        print(
            "ERROR:",
            type(exc).__name__,
            str(exc),
        )"""

'WARD_TESTS = [\n    "Take me to Ward 1.",\n    "Please bring me to Ward 0001.",\n    "I need Ward 27.",\n    "Take me to Ward 1234.",\n    "Please guide me to Ward 5000.",\n    "Take me to the ward.",\n    "Take me to Ward 5001.",\n]\n\n\nfor text in WARD_TESTS:\n\n    print(\n        "\n"\n        + "=" * 70\n    )\n\n    print(\n        "INPUT:",\n        text,\n    )\n\n    try:\n\n        result = (\n            resolver.resolve_destination(\n                text\n            )\n        )\n\n        print(\n            result.model_dump_json(\n                indent=2\n            )\n        )\n\n    except Exception as exc:\n\n        print(\n            "ERROR:",\n            type(exc).__name__,\n            str(exc),\n        )'

In [30]:
"""DEPARTMENT_TESTS = [
    "I need an X-ray.",
    "Where do I get a picture of my bones?",
    "I need to collect my prescribed medicine.",
    "Where is the lift?",
    "I need to use the toilet.",
    "Please take me back to the front desk.",
]


for text in DEPARTMENT_TESTS:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "INPUT:",
        text,
    )

    try:

        result = (
            resolver.resolve_destination(
                text
            )
        )

        print(
            result.model_dump_json(
                indent=2
            )
        )

    except Exception as exc:

        print(
            "ERROR:",
            type(exc).__name__,
            str(exc),
        )"""

'DEPARTMENT_TESTS = [\n    "I need an X-ray.",\n    "Where do I get a picture of my bones?",\n    "I need to collect my prescribed medicine.",\n    "Where is the lift?",\n    "I need to use the toilet.",\n    "Please take me back to the front desk.",\n]\n\n\nfor text in DEPARTMENT_TESTS:\n\n    print(\n        "\n"\n        + "=" * 70\n    )\n\n    print(\n        "INPUT:",\n        text,\n    )\n\n    try:\n\n        result = (\n            resolver.resolve_destination(\n                text\n            )\n        )\n\n        print(\n            result.model_dump_json(\n                indent=2\n            )\n        )\n\n    except Exception as exc:\n\n        print(\n            "ERROR:",\n            type(exc).__name__,\n            str(exc),\n        )'

In [31]:
"""EDGE_CASES = [
    "",
    "Take me to the ward.",
    "Take me somewhere to eat.",
    "Take me to Starbucks.",
    "I need to go there.",
    "Take me to Ward 5001.",
]


for text in EDGE_CASES:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "INPUT:",
        repr(text),
    )

    try:

        result = (
            resolver.resolve_destination(
                text
            )
        )

        print(
            result.model_dump_json(
                indent=2
            )
        )

    except Exception as exc:

        print(
            "ERROR:",
            type(exc).__name__,
            str(exc),
        )"""

'EDGE_CASES = [\n    "",\n    "Take me to the ward.",\n    "Take me somewhere to eat.",\n    "Take me to Starbucks.",\n    "I need to go there.",\n    "Take me to Ward 5001.",\n]\n\n\nfor text in EDGE_CASES:\n\n    print(\n        "\n"\n        + "=" * 70\n    )\n\n    print(\n        "INPUT:",\n        repr(text),\n    )\n\n    try:\n\n        result = (\n            resolver.resolve_destination(\n                text\n            )\n        )\n\n        print(\n            result.model_dump_json(\n                indent=2\n            )\n        )\n\n    except Exception as exc:\n\n        print(\n            "ERROR:",\n            type(exc).__name__,\n            str(exc),\n        )'

In [32]:
def ask_visitor(text: str):
    try:
        decision = resolver.resolve_destination(text)

        print(
            decision.model_dump_json(
                indent=2
            )
        )

        return decision

    except Exception as exc:
        print(
            "ERROR:",
            type(exc).__name__,
            str(exc),
        )

        return None

In [33]:
ask_visitor(
    "Take me to Ward 1234."
)

{
  "intent": "NAVIGATE",
  "destination_type": "WARD",
  "destination": "ward_1234",
  "ward_number": 1234,
  "confidence": 1.0,
  "needs_clarification": false,
  "candidates": [],
  "visitor_message": "Sure, taking you to Ward 1234."
}


DestinationDecision(intent=<Intent.NAVIGATE: 'NAVIGATE'>, destination_type=<DestinationType.WARD: 'WARD'>, destination='ward_1234', ward_number=1234, confidence=1.0, needs_clarification=False, candidates=[], visitor_message='Sure, taking you to Ward 1234.')